In [1]:
# ============================================
# STEP 1: Overview & Setup
# ============================================
# This script processes building simulation outputs for all buildings in the
# baseline_models directory.
#
# For each building:
# 1. Reads:
#    - energy_results_tot.csv
#    - carbon_results.csv
#    - fixed_cost_results.csv
#    - flex_cost_results.csv
#    - annual_results.csv
#
# 2. Sums all numeric columns in each of the four result files.
#
# 3. Writes those sums into the corresponding row in annual_results.csv:
#    - energy_results_tot.csv  -> "energy" row
#    - carbon_results.csv      -> "carbon" row
#    - fixed_cost_results.csv -> "fixed_cost" row
#    - flex_cost_results.csv  -> "flex_cost" row
#
# 4. Saves the updated annual_results.csv back into each building's TOTAL folder.
#
# The script safely skips buildings with missing files and prints progress.

import os
import pandas as pd

# Root directory containing all building folders
BASELINE_DIR = "/Users/jakegehrung/Desktop/data_raw/baseline_models"

In [2]:
# ============================================
# STEP 2: Helper Function to Sum Columns
# ============================================
# This function:
# - Reads a CSV
# - Drops the first column (Time or label)
# - Returns the sum of all remaining columns

def get_column_sums(file_path):
    df = pd.read_csv(file_path)
    
    # Drop first column (Time or label)
    df_numeric = df.iloc[:, 1:]
    
    # Sum all columns
    col_sums = df_numeric.sum()
    
    return col_sums

In [3]:
# ============================================
# STEP 3: Process Each Building Folder
# ============================================
# Iterates through each building directory and updates annual_results.csv

for building_id in os.listdir(BASELINE_DIR):
    building_path = os.path.join(BASELINE_DIR, building_id)
    
    # Skip if not a directory
    if not os.path.isdir(building_path):
        continue
    
    total_path = os.path.join(building_path, "TOTAL")
    
    if not os.path.exists(total_path):
        print(f"Skipping {building_id}: TOTAL folder not found")
        continue
    
    try:
        # File paths
        energy_file = os.path.join(total_path, "energy_results_tot.csv")
        carbon_file = os.path.join(total_path, "carbon_results.csv")
        fixed_file = os.path.join(total_path, "fixed_cost_results.csv")
        flex_file = os.path.join(total_path, "flex_cost_results.csv")
        annual_file = os.path.join(total_path, "annual_results.csv")
        
        # Check all required files exist
        required_files = [energy_file, carbon_file, fixed_file, flex_file, annual_file]
        if not all(os.path.exists(f) for f in required_files):
            print(f"Skipping {building_id}: Missing one or more required files")
            continue
        
        # Get column sums
        energy_sums = get_column_sums(energy_file)
        carbon_sums = get_column_sums(carbon_file)
        fixed_sums = get_column_sums(fixed_file)
        flex_sums = get_column_sums(flex_file)
        
        # Load annual results
        annual_df = pd.read_csv(annual_file)
        
        # Update rows
        annual_df.loc[annual_df["label"] == "energy", energy_sums.index] = energy_sums.values
        annual_df.loc[annual_df["label"] == "carbon", carbon_sums.index] = carbon_sums.values
        annual_df.loc[annual_df["label"] == "fixed_cost", fixed_sums.index] = fixed_sums.values
        annual_df.loc[annual_df["label"] == "flex_cost", flex_sums.index] = flex_sums.values
        
        # Save updated file
        annual_df.to_csv(annual_file, index=False)
        
        print(f"Processed building {building_id}")
    
    except Exception as e:
        print(f"Error processing {building_id}: {e}")

Processed building 1001664924
Processed building 1001829085
Processed building 1001063639
Processed building 1001829071
Processed building 1234761002
Processed building 1002539407
Processed building 1001063637
Processed building 1001664941
Processed building 1001991633
Processed building 1235057414
Processed building 1001829079
Processed building 1001664922
Processed building 1234761003
Processed building 1001664925
Processed building 1000238907
Processed building 1234761004
Processed building 1002313096
Processed building 1001870878
Processed building 1001664940
Processed building 1234982990
Processed building 1234806523
Processed building 1001870876
Processed building 1001870882
Processed building 1002143357
Processed building 1001951902
Processed building 1234621926
Processed building 1234647955
Processed building 1001906269
Processed building 1001951903
Processed building 1001627539
Processed building 1002143351
Processed building 1001627541
Processed building 1236594950
Processed 

In [4]:
# ============================================
# STEP 4: Optional Validation Check
# ============================================
# This block lets you manually inspect one building to confirm correctness

sample_building = "1001664942"  # Change as needed
sample_path = os.path.join(BASELINE_DIR, sample_building, "TOTAL", "annual_results.csv")

if os.path.exists(sample_path):
    df_check = pd.read_csv(sample_path)
    display(df_check.head())
else:
    print("Sample building not found.")

,label,electricity_exp_1,electricity_tot_1,lpg_tot_1,mineral_wood_tot_1,mains_gas_tot_1,oil_tot_1,wood_logs_tot_1,wood_pellets_tot_1,wood_chips_tot_1,...,electricity_exp_9,electricity_tot_9,lpg_tot_9,mineral_wood_tot_9,mains_gas_tot_9,oil_tot_9,wood_logs_tot_9,wood_pellets_tot_9,wood_chips_tot_9,coal_tot_9
0,energy,0.0,2.741105e+07,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,312452.995306,1.207156e+07,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,carbon,0.0,1.863951e+03,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,21.246804,8.208661e+02,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,fixed_cost,0.0,7.039156e+03,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,28.276996,3.099977e+03,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,flex_cost,0.0,5.901516e+03,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,21.069862,2.360399e+03,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
